# CLINIX-AI: train CT and MRI models and run the evaluation (Google Colab)

Runtime > Change runtime type > **GPU** before you start. Run the cells in order.
What you get at the end: `ct.pt` / `mri.pt` model files (+ meta json) and `results_*.json` with the numbers for your report.

You need: a free Kaggle account and a `kaggle.json` token (Kaggle > Settings > API > Create New Token),
and `clinix-ai.zip` (the project zip) uploaded to the Colab file panel.

Honest limits: CT/MRI models classify single 2D images, not 3D volumes. Public datasets can put images
from the same patient in both train and test, which flatters the results. The X-ray evaluation is external
(PadChest-trained weights tested on NIH images).

In [ ]:
!unzip -q -o clinix-ai.zip
!pip -q install torchxrayvision scikit-learn scipy pydicom opencv-python-headless kaggle
import torch; print('GPU:', torch.cuda.is_available())

## 1. Kaggle login (upload kaggle.json when asked)

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 2. Download the datasets

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data/mri --unzip -q
!kaggle datasets download -d mohamedhanyyy/chest-ctscan-images -p data/ct --unzip -q
!kaggle datasets download -d nih-chest-xrays/sample -p data/nih --unzip -q
!find data -maxdepth 3 -type d | head -40

In [ ]:
import os
def find(root, names):
    for r, d, _ in os.walk(root):
        for x in d:
            if x.lower() in names: return os.path.abspath(os.path.join(r, x))
def findfile(root, name):
    for r, _, f in os.walk(root):
        if name in f: return os.path.abspath(os.path.join(r, name))
MRI_TRAIN, MRI_TEST = find('data/mri', {'training'}), find('data/mri', {'testing'})
CT_TRAIN, CT_VAL, CT_TEST = find('data/ct', {'train'}), find('data/ct', {'valid', 'val'}), find('data/ct', {'test'})
NIH_CSV, NIH_IMGS = findfile('data/nih', 'sample_labels.csv'), find('data/nih', {'images'})
print(MRI_TRAIN, MRI_TEST, CT_TRAIN, CT_VAL, CT_TEST, NIH_CSV, NIH_IMGS, sep='\n')
assert all([MRI_TRAIN, MRI_TEST, CT_TRAIN, CT_VAL, CT_TEST, NIH_CSV, NIH_IMGS]), 'A folder was not found: check the printout above'

## 3. Train the MRI model (brain tumour: glioma / meningioma / pituitary / no tumor)

In [ ]:
!python clinix-ai/evaluation/train_classifier.py --name mri --train {MRI_TRAIN} --test {MRI_TEST} --epochs 6

## 4. Train the CT model (chest CT: adenocarcinoma / large cell / squamous cell / normal)

In [ ]:
!python clinix-ai/evaluation/train_classifier.py --name ct --train {CT_TRAIN} --val {CT_VAL} --test {CT_TEST} --epochs 10 --batch 16

## 5. External X-ray evaluation on the NIH sample (accuracy, calibration, sex/age bias, reliability check)

In [ ]:
!python clinix-ai/evaluation/eval_xray_nih.py --csv {NIH_CSV} --images {NIH_IMGS}

## 6. Download the model files and result numbers

In [ ]:
!cd clinix-ai && zip -q -r ../clinix_results.zip backend/models/*.pt backend/models/*_meta.json evaluation/results_*.json
files.download('clinix_results.zip')

## What to do next (on your laptop)
1. Unzip `clinix_results.zip` into your project folder (it puts the files in `backend/models` and `evaluation`).
2. Restart the server: CT and MRI now appear without "(no model installed yet)".
3. Put the numbers from `evaluation/results_*.json` into the Result column of the implementation document.
4. Set the X-ray calibration temperature from the printout: `$env:CLINIX_TEMPERATURE="<value>"`.